# 基礎編12
このノートブックでは、呼び出したユーザーを識別して処理を行うスマートコントラクトの簡単な例を示します。

## スマートコントラクトの処理の概要
- 呼び出したユーザをgetCallerId()によって識別する。
- ユーザIDをキーにして内蔵キーバリューストアに呼び出し回数を記憶する。
- ユーザIDと呼び出し回数を返す。

## 準備

実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。


In [2]:
var users = [];
for(var i=0; i<=2; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct


## スマートコントラクトのデプロイ

スマートコントラクトを便宜的にfunctionとして記述します。

In [3]:
function basic12(){
    var uid = getCallerId();
    if(uid === 'anonymous'){
        throw "anonymous access not allowed";
    }
    var count = 1+keyValueGet(uid);
    keyValueSet(uid, count);
    return { uid: uid, count: count };
}

上記のスマートコントラクトをデプロイします。

In [4]:
var cid = await deploySmartContract(basic12);
console.log(cid);

c040775665


デプロイしたスマートコントラクトの呼び出しを、上記user0～user2 に許可します。


In [5]:
await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: [users[0].id, users[1].id, users[2].id] });

{
  txno: 701804,
  txid: 'xETsjX2FUGzBHBQhqsw3Vb3qdsJqzwGcCvx5hBRrvBXgHB',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

user0のウォレットで最初の呼び出しを行うと、user0のIDと、呼び出し回数1が返ります。

In [6]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701805,
  txid: 'x9xS5kpf7vKu6yNahYziUK5hwiRo5DTjMyrJD9JJvoyAs',
  status: 'ok',
  value: { uid: 'u73621973', count: 1 }
}


user0のウォレットでもう一度呼び出しを行うと、user0のIDと、呼び出し回数2が返ります。

In [7]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701806,
  txid: 'xseDUsEkTx82NwHfZRnTL8PkfS7uspsT64t8ttErnKT7NB',
  status: 'ok',
  value: { uid: 'u73621973', count: 2 }
}


user0のウォレットでさらに 呼び出しを行うと、user0のIDと、呼び出し回数3が返ります。

In [8]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701807,
  txid: 'x8eVFUgAPeDb7zN8vf7uADYvc8VaebA7UFtErBMptxP8v',
  status: 'ok',
  value: { uid: 'u73621973', count: 3 }
}


user1のウォレットで呼び出しを行うと、別のユーザとなるので、呼び出し回数1が返ります。

In [9]:
var resp = await rpc.call(users[1].wallet, cid);
console.log(resp);

{
  txno: 701808,
  txid: 'xf66vszeCsi4WowARVRSfHNLTzGKX9GCdU9ibZPWhy6NFB',
  status: 'ok',
  value: { uid: 'u28169743', count: 1 }
}


user1のウォレットでもう一度呼び出しを行うと、user1のIDと、呼び出し回数2が返ります。

In [10]:
var resp = await rpc.call(users[1].wallet, cid);
console.log(resp);

{
  txno: 701809,
  txid: 'xxvQ5uwURGpKNTy7Ads8Sokmd4UpvYJrkF76gpVPE2GVy',
  status: 'ok',
  value: { uid: 'u28169743', count: 2 }
}


user0のウォレットで呼び出しを行うと、以前の呼び出しが記憶されており、呼び出し回数4が返ります。

In [11]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701810,
  txid: 'xGvpAF2tGibepqFb4cubqxeSgUPZdt5qR4f8qo7gWnkCQB',
  status: 'ok',
  value: { uid: 'u73621973', count: 4 }
}


user2のウォレットで呼び出しを行うと、最初の呼び出しになるので、呼び出し回数1が返ります。

In [12]:
var resp = await rpc.call(users[2].wallet, cid);
console.log(resp);

{
  txno: 701811,
  txid: 'x9iF5o43q4bY5vAiFcZyH4rrEcodraykQZiYbpQzW7HNz',
  status: 'ok',
  value: { uid: 'u53371386', count: 1 }
}
